# Stanford Cars Training Notebook

This notebook trains a pretrained ResNet50 on Stanford Cars images cropped by bounding boxes, tracks validation accuracy, uses early stopping, and evaluates the best checkpoint on the test split.

In [ ]:
from __future__ import annotations

import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from PIL import Image, ImageFile

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet50_Weights, resnet50
import scipy.io
import h5py
from utils import build_transforms, StanfordCarsDataset, load_annotation_file, extract_to_dataframe, prepare_dataframe
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [ ]:
# Configuration
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TRAIN_ANNO_FILE = Path("./archive/cars_train/cars_train_annos.mat")
TEST_ANNO_FILE = Path("./archive/cars_test/cars_test_annos.mat")
META_FILE = Path("./archive/cars_meta.mat")
TRAIN_IMG_DIR = Path("./archive/cars_train/cars_train")
TEST_IMG_DIR = Path("./archive/cars_test/cars_test")
OUTPUT_DIR = Path("./outputs")

BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
VAL_FRACTION = 0.15
PATIENCE = 5
IMAGE_SIZE = 224
NUM_WORKERS = max(0, min(8, os.cpu_count() or 0))
PIN_MEMORY = DEVICE.type == "cuda"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(SEED)
DEVICE

device(type='cpu')

## Data Loading
Load the Stanford Cars annotations and merge them with the class-name metadata.

In [3]:
train_mat, train_loader_type = load_annotation_file(TRAIN_ANNO_FILE)
test_mat, test_loader_type = load_annotation_file(TEST_ANNO_FILE)
meta_mat, meta_loader_type = load_annotation_file(META_FILE)

train_df_raw = extract_to_dataframe(train_mat, train_loader_type)

test_df_raw = extract_to_dataframe(test_mat, test_loader_type)
print(test_df_raw.head())


class_names = meta_mat["class_names"].squeeze()
meta_df = pd.DataFrame({
    "class_id": np.arange(1, class_names.size + 1),
    "class_name": [cls.item() for cls in class_names],
})

combined_df = pd.merge(train_df_raw, meta_df, left_on="class", right_on="class_id", how="left")
    
# make the last 1000 rows of combined_df the test set
test_df = combined_df.tail(1000).reset_index(drop=True)
train_df = combined_df.head(len(combined_df) - 1000).reset_index(drop=True)    

print(combined_df.head())
print(test_df.head())

   bbox_x1  bbox_y1  bbox_x2  bbox_y2      fname
0       30       52      246      147  00001.jpg
1      100       19      576      203  00002.jpg
2       51      105      968      659  00003.jpg
3       67       84      581      407  00004.jpg
4      140      151      593      339  00005.jpg
   bbox_x1  bbox_y1  bbox_x2  bbox_y2  class      fname  class_id  \
0       39      116      569      375     14  00001.jpg        14   
1       36      116      868      587      3  00002.jpg         3   
2       85      109      601      381     91  00003.jpg        91   
3      621      393     1484     1096    134  00004.jpg       134   
4       14       36      133       99    106  00005.jpg       106   

                            class_name  
0                  Audi TTS Coupe 2012  
1                  Acura TL Sedan 2012  
2           Dodge Dakota Club Cab 2007  
3     Hyundai Sonata Hybrid Sedan 2012  
4  Ford F-450 Super Duty Crew Cab 2012  
   bbox_x1  bbox_y1  bbox_x2  bbox_y2  class 

In [ ]:


train_df = prepare_dataframe(combined_df)
test_df = prepare_dataframe(test_df)

label_encoder = LabelEncoder()
label_encoder.fit(train_df["class_name"])

train_split, val_split = train_test_split(
    train_df,
    test_size=VAL_FRACTION,
    random_state=SEED,
    stratify=train_df["class_name"],
)

print(f"Train split: {len(train_split)}")
print(f"Validation split: {len(val_split)}")
print(f"Test split: {len(test_df)}")
print(f"Classes: {len(label_encoder.classes_)}")

Train split: 6922
Validation split: 1222
Test split: 1000
Classes: 196


## Data Preparation
Clean the annotation tables, encode labels, and split into train and validation sets.

In [5]:
train_transform, eval_transform = build_transforms(IMAGE_SIZE)
train_dataset = StanfordCarsDataset(train_split, TRAIN_IMG_DIR, label_encoder, transform=train_transform)
val_dataset = StanfordCarsDataset(val_split, TRAIN_IMG_DIR, label_encoder, transform=eval_transform)
test_dataset = StanfordCarsDataset(test_df, TEST_IMG_DIR, label_encoder, transform=eval_transform)

print(len(train_dataset), len(val_dataset), len(test_dataset))

6922 1222 1000


## Dataset and Image Pipeline
Define image cropping, transforms, and the custom PyTorch Dataset.

In [6]:
def build_model(num_classes: int) -> nn.Module:
    try:
        model = resnet50(weights=ResNet50_Weights.DEFAULT)
    except Exception as error:
        print(f"Could not load pretrained ResNet50 weights ({error}). Falling back to weights=None.")
        model = resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def run_epoch(model: nn.Module, dataloader: DataLoader, criterion: nn.Module, optimizer=None, scaler=None):
    is_training = optimizer is not None
    model.train(mode=is_training)

    running_loss = 0.0
    running_correct = 0
    total = 0

    context = torch.enable_grad() if is_training else torch.no_grad()
    with context:
        for batch in dataloader:
            images = batch["image"].to(DEVICE, non_blocking=True)
            labels = batch["label"].to(DEVICE, non_blocking=True)

            if is_training:
                optimizer.zero_grad(set_to_none=True)

            with autocast(enabled=DEVICE.type == "cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)

            if is_training:
                if scaler is not None and scaler.is_enabled():
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()

            predictions = outputs.argmax(dim=1)
            running_correct += (predictions == labels).sum().item()
            total += labels.size(0)
            running_loss += loss.item() * labels.size(0)

    return running_loss / max(1, total), running_correct / max(1, total)

class EarlyStopping:
    def __init__(self, patience: int, checkpoint_path: Path) -> None:
        self.patience = patience
        self.checkpoint_path = checkpoint_path
        self.best_score = -float("inf")
        self.bad_epochs = 0

    def step(self, score: float, model: nn.Module, epoch: int, label_encoder: LabelEncoder) -> bool:
        if score > self.best_score:
            self.best_score = score
            self.bad_epochs = 0
            self.save_checkpoint(model, epoch, label_encoder)
            return False
        self.bad_epochs += 1
        return self.bad_epochs >= self.patience

    def save_checkpoint(self, model: nn.Module, epoch: int, label_encoder: LabelEncoder) -> None:
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "label_classes": label_encoder.classes_,
        }, self.checkpoint_path)



## Training
Fine-tune ResNet50 with mixed precision, validation tracking, and early stopping.

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

model = build_model(num_classes=len(label_encoder.classes_)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.5)
scaler = GradScaler(enabled=DEVICE.type == "cuda")
checkpoint_path = OUTPUT_DIR / "best_resnet50_stanford_cars.pt"
early_stopper = EarlyStopping(patience=PATIENCE, checkpoint_path=checkpoint_path)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer=optimizer, scaler=scaler)
    val_loss, val_acc = run_epoch(model, val_loader, criterion)
    scheduler.step(val_acc)

    print(
        f"Epoch {epoch:03d}/{EPOCHS:03d} | "
        f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f}"
    )

    if early_stopper.step(val_acc, model, epoch, label_encoder):
        print(f"Early stopping triggered after {epoch} epochs.")
        break

best_checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
model.load_state_dict(best_checkpoint["model_state_dict"])
print(f"Best model checkpoint: {checkpoint_path}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\nikhi/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100.0%
C:\Users\nikhi\AppData\Local\Temp\ipykernel_15456\409809462.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=DEVICE.type == "cuda")


## Evaluation
Run the best checkpoint on the test set and export the confusion matrix and classification report.